# 05 — Tạo submission public test: PhoWhisper-medium fine-tune (notebook 04) + Qwen3-4B

Dùng nguyên pipeline của team (`python -m lifemate.run`), chỉ thay `--asr-model` bằng model ASR đã fine-tune.

**Độ chính xác số:** đúng dtype gốc — PhoWhisper fine-tune **float32**, Qwen3-4B **bfloat16** (không fp16, không FP8/int8).
**VRAM:** mục tiêu GPU 24 GB của BTC (A30) — 1 GPU chứa cả hai model. Kaggle: chọn **GPU T4 x2** (16 GB/GPU, T4 không có bf16 phần cứng nên chậm hơn A30): LLM ở `cuda:0`, ASR ở `cuda:1`.
**Internet On** (tải Qwen3-4B lần đầu).
**Input:**
1. Dataset data BTC (`train+public_test`).
2. Dataset model ASR: thư mục `phowhisper_medium_ft_v1_fp32` (có `config.json`, `model.safetensors`, tokenizer, `preprocessor_config.json`).
3. Dataset source `lifemate_src` (giải nén từ `lifemate_src.zip`, có `src/lifemate/run.py` bản đã sửa dtype). Bắt buộc: `main` trên GitHub chưa có bản sửa này.

Output: `/kaggle/working/lifemate/public_asr_ft_v1/submission.jsonl` — file nộp lên portal BTC.

In [ ]:
%pip install -q "transformers>=4.51.0,<5" "accelerate>=1.0,<2" "safetensors>=0.5,<1"

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
import torch

RUN_DEV_CHECK = True     # chạy trước 48 mẫu pilot dev (có nhãn) để so điểm; False = vào thẳng public
LLM_MODEL = "Qwen/Qwen3-4B"
INPUT = Path(os.environ.get("LIFEMATE_INPUT", "/kaggle/input"))          # máy khác: đặt biến môi trường
WORK = Path(os.environ.get("LIFEMATE_WORK", "/kaggle/working/lifemate")); WORK.mkdir(parents=True, exist_ok=True)

src = sorted({p.parents[2] for p in INPUT.rglob("src/lifemate/run.py")})
assert len(src) == 1, f"Cần đúng 1 source lifemate trong Input, thấy {src}"
SOURCE_ROOT = src[0]
assert 'torch_dtype="auto"' in (SOURCE_ROOT / "src/lifemate/models.py").read_text(encoding="utf-8"), \
    "Source chưa có bản sửa dtype gốc (fp32/bf16) — dùng lifemate_src.zip mới"
data = sorted({p.parent.parent for p in INPUT.rglob("train.jsonl")
               if p.parent.name == "train" and (p.parent.parent / "public_test/public_test.jsonl").is_file()})
asr = sorted({p.parent for p in INPUT.rglob("config.json")
              if (p.parent / "model.safetensors").is_file() and (p.parent / "preprocessor_config.json").is_file()})
print("source:", SOURCE_ROOT, "\ndata:", data, "\nasr:", asr)
assert len(data) == 1 and len(asr) == 1, "Kiểm tra lại Input: cần đúng 1 dataset data và 1 thư mục model ASR"
DATA_ROOT, ASR_MODEL = data[0], asr[0]
ENV = dict(os.environ, PYTHONPATH=str(SOURCE_ROOT / "src"), PYTHONHASHSEED="42")
assert torch.cuda.is_available(), "Bật GPU trong Session options"
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU{i}: {p.name} {p.total_memory/2**30:.1f} GB | bf16 phần cứng: {p.major >= 8}")
# 1 GPU >= 20 GB (A30/A10/3090/4090...): chung 1 GPU như BTC. 2 GPU nhỏ (Kaggle T4 x2): tách ra.
LLM_DEVICE = "cuda:0"
ASR_DEVICE = "cuda:1" if N_GPU > 1 and torch.cuda.get_device_properties(0).total_memory < 20 * 2**30 else "cuda:0"
print("LLM ->", LLM_DEVICE, "| ASR ->", ASR_DEVICE)
DEVICES = ["--asr-device", ASR_DEVICE, "--llm-device", LLM_DEVICE]

import threading
PEAK = {}
def _watch_vram(stop):
    # Đỉnh VRAM theo nvidia-smi (gồm cả CUDA context) — để biết có vừa 24 GB hay không.
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout
        for line in out.strip().splitlines():
            i, mb = map(int, line.split(","))
            PEAK[i] = max(PEAK.get(i, 0), mb)
        stop.wait(2)

def run_module(module, *args):
    stop = threading.Event(); t = threading.Thread(target=_watch_vram, args=(stop,), daemon=True); t.start()
    try:
        subprocess.run([sys.executable, "-m", "lifemate." + module, *map(str, args)], env=ENV, check=True)
    finally:
        stop.set(); t.join()
        if module == "run":
            print("Đỉnh VRAM (MB):", PEAK, "| tổng:", sum(PEAK.values()))

In [ ]:
# Kiểm tra nhanh: model ASR nạp được bằng class của team và chép đúng 1 câu train
sys.path.insert(0, str(SOURCE_ROOT / "src"))
from lifemate.models import PhoWhisper
row = json.loads(open(DATA_ROOT / "train/train.jsonl", encoding="utf-8").readline())
m = PhoWhisper(str(ASR_MODEL), ASR_DEVICE)
assert m.model.dtype == torch.float32, m.model.dtype
t0 = time.time(); hyp = m.transcribe(DATA_ROOT / "train" / row["audio"], 30)
print(f"params {m.params:,} | dtype {m.model.dtype} | {time.time()-t0:.1f}s\nGT : {row['audio_text']}\nASR: {hyp}")
del m; torch.cuda.empty_cache()

In [ ]:
# (Tùy chọn) Pilot dev 48 mẫu có nhãn: điểm end-to-end với ASR fine-tune
if RUN_DEV_CHECK:
    AUDIT = WORK / "data_audit"
    if not (AUDIT / "pilot_ids.json").exists():
        run_module("audit", "--data-root", DATA_ROOT, "--output", AUDIT)
    PILOT = WORK / "pilot_asr_ft_v1"
    if not PILOT.exists():
        run_module("run", "--data-root", DATA_ROOT, "--split", "train", "--ids", AUDIT / "pilot_ids.json",
                   "--asr-model", ASR_MODEL, "--llm-model", LLM_MODEL, *DEVICES, "--output", PILOT)
    run_module("evaluate", "--data-root", DATA_ROOT, "--run", PILOT)
    r = json.loads((PILOT / "evaluation.json").read_text())
    print(json.dumps({k: r.get(k) for k in ["samples", "state_match_rate", "mean_exact_call_f1", "asr",
                                             "latency_seconds", "stop_reasons"]}, ensure_ascii=False, indent=1))

In [ ]:
# ===== Public test -> submission.jsonl =====
PUBLIC_RUN = WORK / "public_asr_ft_v1"
assert not PUBLIC_RUN.exists() or not (PUBLIC_RUN / "submission.jsonl").exists(), "Đã có submission; đổi tên PUBLIC_RUN nếu muốn chạy lại"
t0 = time.time()
run_module("run", "--data-root", DATA_ROOT, "--split", "public_test",
           "--asr-model", ASR_MODEL, "--llm-model", LLM_MODEL, *DEVICES, "--output", PUBLIC_RUN)
print(f"public run: {(time.time()-t0)/60:.1f} phút")

In [ ]:
# ===== Kiểm tra định dạng bằng công cụ BTC + đủ id =====
validate_env = dict(ENV, PYTHONPATH=str(DATA_ROOT / "public_test"))
subprocess.run([sys.executable, "-m", "zaic_tools", "--split", "public_test", "--root", str(DATA_ROOT),
                "--validate", str(PUBLIC_RUN / "submission.jsonl")], env=validate_env, check=True)
rows = [json.loads(l) for l in (PUBLIC_RUN / "submission.jsonl").read_text(encoding="utf-8").splitlines()]
expected = [json.loads(l)["id"] for l in (DATA_ROOT / "public_test/public_test.jsonl").read_text(encoding="utf-8").splitlines()]
assert len(rows) == len(expected) and {r["id"] for r in rows} == set(expected), (len(rows), len(expected))
assert all(set(r) >= {"id", "tool_calls", "state_hash"} for r in rows)
print(f"OK: {len(rows)} dòng, đủ id. File nộp: {PUBLIC_RUN / 'submission.jsonl'}")
print("Số mẫu không gọi tool:", sum(not r["tool_calls"] for r in rows))